# 02.5 — Does curvature through a *trained* chart decoder recover a known answer?

Stage 2's Arm B is new, untested code that takes **second derivatives of a trained neural
network**. Its case (D-09) is that a learned decoder averages sampling noise away and is then
differentiated exactly, so the neighbourhood radius `r/R` that broke the point-cloud estimator at
`d = 20` never enters its error at all. Its counter-risk, from the same decision, is that a
network's second derivative can wiggle where the true manifold does not — and the converse, that a
reconstruction-trained decoder can smooth genuine curvature flat while reconstructing every point
beautifully. On real data those two look identical. Only a manifold with a closed-form answer
separates them.

**The trap this notebook exists to expose, stated concretely.** If the truth is `y = a x²` and the
decoder learns `y = 0.7 a x²`, reconstruction error stays tiny wherever the sampled `x` sit near
zero, while the second derivative is `1.4a` instead of `2a` — 30% curvature attenuation with *no
reconstruction signal at all*. **Reconstruction quality and curvature agreement are therefore two
separate numbers here and are never combined.** A low reconstruction error printed beside a poor
curvature score is the finding, not a bug to hide.

Nothing here is a gate. Stage 2's gate is fixed only by the phase's ratified stage-2
pre-registration document and evaluated by `notebooks/diagnostics/cae_local_regate_run.py`. Every
number below is a sanity read-out. Nothing is cached, nothing is written, no sealed 02.2 fit is
loaded — this notebook trains its own model and fits in about a minute.

## §1. Setup

In [ ]:
import sys
import time
from pathlib import Path

# import pu_manifold exactly as the other notebooks do -- relative, never from src/effdim/
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib.pyplot as plt
import numpy as np
import torch

from pu_manifold import cae, chart_curvature, curvature_probe

SEED = 0               # torch / split seed, as in 02.2's notebook
FIXTURE_SEED = 20260807  # the roll's own random_state, as the rest of phase 02.5 uses
N_POINTS = 3000
CHART_DIM = 2          # the Swiss roll's true intrinsic dimension
EMBED_DIM = 8
N_CHARTS = 8
HIDDEN = [64, 64]
K_BASELINE = 30        # the raw-point baseline's neighbourhood size, as in plan 02.5-05

print(f"torch {torch.__version__}  numpy {np.__version__}")
print(f"cae             : {cae.__file__}")
print(f"chart_curvature : {chart_curvature.__file__}")
print(f"curvature_probe : {curvature_probe.__file__}")
print(f"convention      : {chart_curvature.CURVATURE_CONVENTION} "
      f"(H = tr_g(II), matching curvature_probe: {curvature_probe.CURVATURE_CONVENTION})")

## §2. The Swiss roll, and its curvature in closed form

`curvature_probe.make_swiss_roll_fixture` is called rather than transcribed, so this notebook
uses the same preprocessing path the rest of the phase does: `make_swiss_roll` with
`noise=0.0` and a fixed `random_state`, centred, then divided by **one global scalar** standard
deviation. One scalar means the shape is preserved exactly and only the overall size changes —
and curvature, having units of inverse length, is multiplied by that same scalar, which is what
`swiss_roll_analytic_H_scaled` handles inside the fixture.

The mean curvature **vector** is needed too, not just its norm, because direction and amplitude
are distinct failure modes and only a vector can tell them apart. The fixture supplies the norm;
the unit direction is the one thing derived here. The roll is a ruled generalized cylinder over
the planar Archimedean spiral `c(t) = (t·cos t, t·sin t)`, whose ruling direction is exactly
straight, so the surface's mean curvature vector *is* that plane curve's own curvature vector,
`H = P_⊥c'' / |c'|²`, lying entirely in the x–z plane. The cell below pins the norm of that
derived vector against the module's sealed `H_norm`, so nothing about the direction can drift
away from the ground truth the rest of the phase gates on.

Colour is the roll's own arc-length parameter `t`, so the same colour always means the same
place on the sheet: colour bands staying in order means the surface stayed in order.

In [ ]:
fx = curvature_probe.make_swiss_roll_fixture(n=N_POINTS, seed=FIXTURE_SEED)
X, t, H_true_norm, global_std = fx["X"], fx["t"], fx["H_norm"], fx["global_std"]

# analytic mean curvature VECTOR: the spiral's own curvature vector, in the x-z plane
ct, st = np.cos(t), np.sin(t)
d1 = np.stack([ct - t * st, st + t * ct], axis=1)          # c'(t)
d2 = np.stack([-2 * st - t * ct, 2 * ct - t * st], axis=1)  # c''(t)
speed2 = np.sum(d1 * d1, axis=1)
k_vec = (d2 - (np.sum(d2 * d1, axis=1) / speed2)[:, None] * d1) / speed2[:, None]
H_true = np.zeros((N_POINTS, 3))
H_true[:, 0] = k_vec[:, 0] * global_std
H_true[:, 2] = k_vec[:, 1] * global_std

pin = float(np.abs(np.linalg.norm(H_true, axis=1) - H_true_norm).max())
print(f"X {X.shape}   global_std = {global_std:.4f}")
print(f"analytic ||H||  range [{H_true_norm.min():.4f}, {H_true_norm.max():.4f}]")
print(f"derived H vector vs the module's sealed H_norm, max abs diff = {pin:.2e}")

rng = np.random.default_rng(SEED)
perm = rng.permutation(N_POINTS)
n_train = int(0.8 * N_POINTS)
train_idx, holdout_idx = perm[:n_train], perm[n_train:]
x_all = torch.tensor(X, dtype=torch.float32)
x_train, x_holdout = x_all[train_idx], x_all[holdout_idx]
print(f"train {len(train_idx)}   holdout {len(holdout_idx)}")

VIEW = dict(elev=12, azim=-78)  # looks along the roll's extrusion axis, so the spiral shows

fig = plt.figure(figsize=(10, 4.5))
ax = fig.add_subplot(1, 2, 1, projection="3d")
ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=t, cmap="viridis", s=4)
ax.view_init(**VIEW)
ax.set_title("Swiss roll (input), coloured by arc-length t")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax2 = fig.add_subplot(1, 2, 2)
ax2.scatter(X[:, 0], X[:, 2], c=t, cmap="viridis", s=4)
ax2.set_title("same points, x-z plane (the spiral)")
ax2.set_xlabel("x"); ax2.set_ylabel("z"); ax2.set_aspect("equal")
plt.tight_layout(); plt.show()

## §3. Train a Chart Auto-Encoder from scratch

Same `cae.ChartAutoEncoder` and `cae.train_cae` as phase 02.2, at `chart_dim = 2` — the roll's
true intrinsic dimension. **Every hyperparameter is reused verbatim from
`notebooks/02.2_swiss_roll_cae_check.ipynb`** (8 charts, embedding dimension 8, two hidden layers
of 64, `lr=1e-3`, `weight_decay=1e-4`, batch 64, 300 epochs with patience 25, eq.-4 Lipschitz
weight `1e-3`, and 20 epochs of eq.-5 FPS pre-training) so that no new tunable is introduced by a
sanity check. The FPS pre-training is not optional here: without it a multi-chart model never
activates its second chart, and this would then be testing a degenerate atlas.

`activation="silu"` is load-bearing rather than stylistic. A piecewise-linear activation has a
second derivative that is a sum of Dirac deltas at its kinks, numerically **exactly zero**
everywhere autodiff evaluates it — so the whole second fundamental form would come back as `0.0`
and read as a perfectly flat manifold instead of raising. `chart_curvature.assert_c2_activation`
refuses to differentiate such a decoder, and it is called on every path below.

The matched baseline for reconstruction is trained in the same cell: `cae.PlainAutoEncoder` at
the same 2-D bottleneck, same width, same depth, same training protocol.

In [ ]:
BASE_CFG = dict(seed=SEED, lr=1e-3, weight_decay=1e-4, batch=64, max_epochs=300,
                early_stop_patience=25, early_stop_min_delta=1e-4)

wall0 = time.time()

torch.manual_seed(SEED)
model = cae.ChartAutoEncoder(in_dim=3, embed_dim=EMBED_DIM, chart_dim=CHART_DIM,
                             n_charts=N_CHARTS, hidden=HIDDEN, activation="silu")
fit = cae.train_cae(model, x_train, {**BASE_CFG, "n_charts": N_CHARTS,
                                     "fps_pretrain_epochs": 20,
                                     "lip_weight": 1e-3, "lip_every_n_steps": 1})
model.eval()

torch.manual_seed(SEED)
plain = cae.PlainAutoEncoder(3, CHART_DIM, hidden=tuple(HIDDEN), activation="silu")
plain_fit = cae.train_plain_ae(plain, x_train, dict(BASE_CFG))
plain.eval()

train_wall = time.time() - wall0
print(f"CAE     epochs_run = {fit['epochs_run']:3d}  early_stopped = {fit['early_stopped']}")
print(f"plain   epochs_run = {plain_fit['epochs_run']:3d}  early_stopped = {plain_fit['early_stopped']}")
print(f"activation recorded on the model = "
      f"{chart_curvature.assert_c2_activation(model)!r}  (C2, so twice-differentiable)")
print(f"training wall-clock = {train_wall:.1f} s")

main_history = [h for h in fit["history"] if h["stage"] == "main"]
plt.figure(figsize=(5, 3))
plt.plot([h["epoch"] for h in main_history], [h["total"] for h in main_history])
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("CAE training curve")
plt.tight_layout(); plt.show()

## §4. Reconstruction first — establishing the premise the next section tests

This section answers only one question: **is the reconstruction good?** It is CLAUDE.md's matched
baseline comparison, and it is deliberately reported before any curvature number is computed,
because its whole role here is to set up the trap. If the reconstruction is excellent and the
curvature agreement in §6 is poor, that gap is the finding — and it would be invisible on data
with no known answer.

Each point is encoded, routed to its highest-probability chart, and decoded back to 3-D through
that chart's decoder and the shared embedding decoder (`model.reconstruct`, the same argmax-chart
path phase 02.2 measured), on held-out points the model never trained on.

In [ ]:
with torch.no_grad():
    y_hold = model.reconstruct(x_holdout)
    y_plain = plain(x_holdout)["y"]
    y_all = model.reconstruct(x_all).numpy()

cae_stats = cae.reconstruction_stats(x_holdout.double(), y_hold.double())
plain_stats = cae.reconstruction_stats(x_holdout.double(), y_plain.double())
survival = cae.chart_survival(model, prune_tol=1e-2)
rel_err = float((torch.linalg.vector_norm(x_holdout - y_hold, dim=-1)
                 / torch.linalg.vector_norm(x_holdout, dim=-1)).mean())
recon_ratio = plain_stats["mse_per_dim"] / cae_stats["mse_per_dim"]

print("=== held-out reconstruction ===")
print(f"CAE            mse_per_dim = {cae_stats['mse_per_dim']:.6f}")
print(f"plain AE (d=2) mse_per_dim = {plain_stats['mse_per_dim']:.6f}")
print(f"CAE mean relative error    = {rel_err:.4f}  ({100 * rel_err:.1f}% of point norm)")
print(f"CAE is {recon_ratio:.2f}x better than the matched plain-AE")
print(f"charts surviving           = {survival['n_charts_surviving']} / {survival['n_charts_initial']}")

fig = plt.figure(figsize=(11, 9))
for i, (data, title) in enumerate([(X, "original"), (y_all, "CAE reconstruction")]):
    ax = fig.add_subplot(2, 2, i + 1, projection="3d")
    ax.scatter(data[:, 0], data[:, 1], data[:, 2], c=t, cmap="viridis", s=4)
    ax.view_init(**VIEW); ax.set_title(f"{title} (colour = t)")
    ax.set_xlim(X[:, 0].min(), X[:, 0].max()); ax.set_ylim(X[:, 1].min(), X[:, 1].max())
    ax.set_zlim(X[:, 2].min(), X[:, 2].max())
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax2 = fig.add_subplot(2, 2, i + 3)
    ax2.scatter(data[:, 0], data[:, 2], c=t, cmap="viridis", s=4)
    ax2.set_title(f"{title} — x-z plane"); ax2.set_aspect("equal")
    ax2.set_xlim(X[:, 0].min(), X[:, 0].max()); ax2.set_ylim(X[:, 2].min(), X[:, 2].max())
    ax2.set_xlabel("x"); ax2.set_ylabel("z")
plt.tight_layout(); plt.show()

## §5. Curvature through the chart decoder, and §6 against the analytic answer

**This is the step nothing before it tested.** `chart_curvature.chart_curvature_field` encodes
each point, routes it to the chart the model itself assigns, and differentiates *that chart's*
decoder twice with `torch.func`, exactly — no neighbourhood is formed and no radius appears
anywhere, which is the entire statistical argument for this arm. `metric_condition_number` is
printed because a near-singular pullback metric means the decoder is not an immersion there, and
curvature at such a point is meaningless no matter what the rank statistic says.

Then four numbers on four separate axes, because `H` is **vector-valued** and amplitude
attenuation and orientation error are distinct failure modes with different remedies. A rank
statistic is exactly blind to amplitude: a decoder compressing every magnitude by a constant
factor scores a perfect `1.0` on Spearman.

1. **direction** — cosine similarity against the analytic `H`
2. **magnitude** — `||H_est|| / ||H_true||`, median **and** per-point CV, both required
3. **calibration** — regress `||H_est||` on `||H_true||`; a well-behaved estimator gives slope
   `a ≈ 1`, intercept `b ≈ 0`
4. **rank** — Spearman, reported *alongside* the three above and never instead of them

The matched baseline is `curvature_probe.centroid_mean_curvature(X, k=30, d=2)`, the raw-point
estimator on the identical fixture. For calibration: `02.5-NOTE-high-d-curvature-approaches.md`
§1a records that estimator at `d=2` giving median ratio `0.905` at CV `0.250` — "a mild
underestimate with modest scatter, a correctable signature". Its measured stage-1 Spearman on
this fixture was `0.6712` (plan 02.5-05), which missed that notebook's own `0.90` bar, so it is a
baseline that *works* rather than one that is perfect — the honest comparison, stated plainly.

In [ ]:
model.double()  # second derivatives are exactly where float32 noise shows; the module refuses float32
curv0 = time.time()
field = chart_curvature.chart_curvature_field(model, x_all.double())
curv_wall = time.time() - curv0

H_chart = field["H_vec"].numpy()
h_chart = field["H_norm"].numpy()
cond = field["metric_condition_number"].numpy()
cond_max = float(cond.max())

base0 = time.time()
H_raw = curvature_probe.centroid_mean_curvature(X, k=K_BASELINE, d=CHART_DIM)
h_raw = curvature_probe.mean_curvature_norm(H_raw)
base_wall = time.time() - base0

rho_chart = curvature_probe.spearman_gate_statistic(h_chart, H_true_norm)
rho_raw = curvature_probe.spearman_gate_statistic(h_raw, H_true_norm)
mre_chart = curvature_probe.median_relative_error(h_chart, H_true_norm)
mre_raw = curvature_probe.median_relative_error(h_raw, H_true_norm)
fid_chart = chart_curvature.curvature_fidelity_report(H_chart, H_true)
fid_raw = chart_curvature.curvature_fidelity_report(H_raw, H_true)

print(f"chart-decoder field: {field['n_charts_used']} charts used, "
      f"{curv_wall:.1f} s   (raw-point baseline {base_wall:.1f} s)")
print(f"metric condition number: median {np.median(cond):.2f}  max {cond_max:.2f}")
print()
hdr = f"{'':<24}{'chart decoder':>16}{'raw points (k=30)':>20}"
print(hdr); print("-" * len(hdr))
for label, a, b in [
    ("1. direction  cos", fid_chart["median_cosine_similarity"], fid_raw["median_cosine_similarity"]),
    ("2. magnitude  median", fid_chart["median_magnitude_ratio"], fid_raw["median_magnitude_ratio"]),
    ("2. magnitude  CV", fid_chart["magnitude_ratio_cv"], fid_raw["magnitude_ratio_cv"]),
    ("3. calib. slope a", fid_chart["calibration_slope"], fid_raw["calibration_slope"]),
    ("3. calib. intercept b", fid_chart["calibration_intercept"], fid_raw["calibration_intercept"]),
    ("3. calib. R^2", fid_chart["calibration_r2"], fid_raw["calibration_r2"]),
    ("4. rank  Spearman", rho_chart, rho_raw),
    ("   median rel. error", mre_chart, mre_raw),
]:
    print(f"{label:<24}{a:>16.4f}{b:>20.4f}")

### The pictures

Top row, 3-D; bottom row, the x-z plane, where the spiral is unambiguous. Analytic `||H||`,
chart-decoder `||H||`, raw-point `||H||`. **Each panel is normalized to its own 2–98 percentile
range on purpose**, so what is being compared here is *spatial pattern* only — amplitude is
deliberately not readable from these colours, because amplitude belongs to the magnitude-ratio and
calibration numbers above and conflating the two is the exact error this notebook is about. The
analytic field is highest at the tight inner turn and falls smoothly outward; a field that bands
along a chart boundary rather than along the spiral is tracking the atlas, not the geometry.

Then the two identity-line scatters, on shared axes. This is where amplitude *is* readable: points
on the diagonal mean the estimate has the right size, a flattened cloud means the rank statistic
has nothing to work with, and a systematically shallow slope is the smoothing failure mode.

In [ ]:
panels = [("analytic ||H||", H_true_norm), ("chart-decoder ||H||", h_chart),
          ("raw-point ||H|| (k=30)", h_raw)]

fig = plt.figure(figsize=(15, 8.5))
for j, (title, vals) in enumerate(panels):
    lo, hi = np.percentile(vals, [2, 98])
    ax = fig.add_subplot(2, 3, j + 1, projection="3d")
    ax.scatter(X[:, 0], X[:, 1], X[:, 2], c=vals, cmap="magma", s=4, vmin=lo, vmax=hi)
    ax.view_init(**VIEW); ax.set_title(f"{title}\n(own 2-98 pct scale)", fontsize=10)
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax2 = fig.add_subplot(2, 3, j + 4)
    sc = ax2.scatter(X[:, 0], X[:, 2], c=vals, cmap="magma", s=4, vmin=lo, vmax=hi)
    ax2.set_title(f"{title} — x-z plane", fontsize=10); ax2.set_aspect("equal")
    ax2.set_xlabel("x"); ax2.set_ylabel("z")
    fig.colorbar(sc, ax=ax2, fraction=0.046)
plt.tight_layout(); plt.show()

lim = (0.0, float(max(H_true_norm.max(), np.percentile(h_chart, 99), np.percentile(h_raw, 99))) * 1.05)
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, (name, vals, rho) in zip(axes, [("chart decoder", h_chart, rho_chart),
                                        ("raw points (k=30)", h_raw, rho_raw)]):
    ax.scatter(H_true_norm, vals, s=4, alpha=0.35)
    ax.plot(lim, lim, "k--", lw=1, label="identity")
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_aspect("equal")
    ax.set_xlabel("analytic ||H||"); ax.set_ylabel("estimated ||H||")
    ax.set_title(f"{name}   (Spearman {rho:.4f})")
    ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

## §7. Read-out

**The pattern to look for is a PASS on the first line beside a FAIL on the second.** That
combination — excellent reconstruction, poor curvature agreement — is D-09's counter-risk showing
up on a manifold where the answer is known, and it is the single most important thing this
notebook can find. It says the decoder's second derivative does not follow the manifold's even
when the decoder's *values* do, and it is invisible on data with no known answer.

The two are printed as separate lines and are never combined into one verdict. If the
chart-decoder arm loses to the raw-point baseline, that is printed plainly: a model that is not
better than the baseline on a manifold it was designed for is a real result about the model, and
D-09 makes "the CAE is not needed here" a legitimate, reportable outcome that routes to the
Ollivier-Ricci fork rather than a phase failure.

In [ ]:
beats_plain = recon_ratio > 1.0
curv_ok = rho_chart > 0.90
beats_raw = rho_chart > rho_raw
cond_ok = bool(np.isfinite(cond).all()) and cond_max < 1e6

print(f"1. reconstruction beats the matched plain-AE      : {beats_plain}"
      f"   (ratio {recon_ratio:.2f}x, CAE rel. err {100 * rel_err:.1f}%)")
print(f"2. chart-decoder curvature Spearman > 0.90        : {curv_ok}"
      f"   (rho = {rho_chart:.4f})")
print(f"3. chart-decoder beats the raw-point baseline     : {beats_raw}"
      f"   (chart {rho_chart:.4f}  vs  raw {rho_raw:.4f})")
print(f"4. metric condition number finite and < 1e6       : {cond_ok}"
      f"   (max {cond_max:.2f})")
print()
print("Reconstruction and curvature agreement are two separate numbers above and are NOT combined.")
print()

if beats_plain and not curv_ok:
    print(
        "Read-out: the chart auto-encoder reconstructs the Swiss roll well "
        f"({100 * rel_err:.1f}% relative error, {recon_ratio:.2f}x the matched plain-AE, "
        f"{survival['n_charts_surviving']}/{survival['n_charts_initial']} charts surviving) and its "
        f"pullback metric is well conditioned everywhere (cond <= {cond_max:.0f}, so it is a genuine "
        "immersion and its curvature is meaningful), yet curvature taken exactly through that trained "
        f"decoder agrees with the closed-form answer at Spearman {rho_chart:.4f} against the raw-point "
        f"estimator's {rho_raw:.4f} -- direction is largely recovered (median cosine "
        f"{fid_chart['median_cosine_similarity']:.3f}) while amplitude is not (median ratio "
        f"{fid_chart['median_magnitude_ratio']:.3f} at CV {fid_chart['magnitude_ratio_cv']:.3f}, "
        f"calibration slope {fid_chart['calibration_slope']:.3f} with R^2 "
        f"{fid_chart['calibration_r2']:.3f}), which is exactly D-09's counter-risk: a good "
        "reconstruction is no evidence at all about a second derivative."
    )
elif curv_ok and beats_raw:
    print(
        f"Read-out: curvature through the trained chart decoder recovers the known answer "
        f"(Spearman {rho_chart:.4f}, median cosine {fid_chart['median_cosine_similarity']:.3f}, median "
        f"magnitude ratio {fid_chart['median_magnitude_ratio']:.3f} at CV "
        f"{fid_chart['magnitude_ratio_cv']:.3f}) and beats the raw-point baseline's "
        f"{rho_raw:.4f} on the same fixture."
    )
else:
    print(
        f"Read-out: reconstruction ratio {recon_ratio:.2f}x, chart-decoder curvature Spearman "
        f"{rho_chart:.4f} vs the raw-point baseline's {rho_raw:.4f}, median magnitude ratio "
        f"{fid_chart['median_magnitude_ratio']:.3f} at CV {fid_chart['magnitude_ratio_cv']:.3f}, "
        f"max metric condition number {cond_max:.2f} -- read the four lines above separately."
    )
print()
print("These are SANITY thresholds, not the phase gate. Stage 2's gate is fixed only by the")
print("phase's ratified stage-2 pre-registration document and evaluated by")
print("notebooks/diagnostics/cae_local_regate_run.py; nothing printed here decides it.")